Computed Fields

In [1]:
from pydantic import BaseModel, Field, EmailStr, AnyUrl, field_validator, model_validator
from typing import List, Dict, Optional, Annotated

In [2]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]

    @field_validator('email')
    @classmethod
    def validate_email(cls, value):
        allowed_domains = ['sbi.com', 'equitas.com']
        domain = value.split('@')[-1]
        if domain not in allowed_domains:
            raise ValueError(f"Email domain must be one of {allowed_domains}")
        return value
    
    @field_validator('name', mode='after')
    @classmethod
    def transform_name(cls, value):
        return value.strip().upper()
    
    @field_validator('age')
    @classmethod
    def validate_age(cls, value):
        if value < 18:
            raise ValueError("Patient must be at least 18 years old")
        return value
    
    @model_validator(mode='after')
    def validate_emergency_contact(cls, model):
        if model.age > 60 and 'emergency_contact' not in model.contact_info:
            raise ValueError("Patients over 60 must have an emergency contact")
        return model

In [3]:
patient_info = {'name': 'benky', 'email': 'benky@sbi.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '65', 'weight': 70.5, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890', 'emergency_contact': '9876543210'}}
patient1 = Patient(**patient_info)
print(patient1)

name='BENKY' email='benky@sbi.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=65 weight=70.5 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890', 'emergency_contact': '9876543210'}


Compute a new field from the exisiting fields

In [4]:
from pydantic import computed_field

In [5]:
class Patient(BaseModel):
    name: str = Annotated[str, Field(max_length=50, title="Patient Name", description="Enter name of the patient in less than 50 characters", examples=["John Doe", "Jane Smith"])]
    email: EmailStr
    linkedin_url: AnyUrl
    age: int = Field(gt=0, lt=120)
    weight: Annotated[float, Field(gt=0, strict=True)]
    height: Annotated[float, Field(gt=0, strict=True)]
    married: Annotated[Optional[bool], Field(default=False, description="Is the patient married?")]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=2)]
    contact_info: Dict[str, str]

    @field_validator('email')
    @classmethod
    def validate_email(cls, value):
        allowed_domains = ['sbi.com', 'equitas.com']
        domain = value.split('@')[-1]
        if domain not in allowed_domains:
            raise ValueError(f"Email domain must be one of {allowed_domains}")
        return value
    
    @field_validator('name', mode='after')
    @classmethod
    def transform_name(cls, value):
        return value.strip().upper()
    
    @field_validator('age')
    @classmethod
    def validate_age(cls, value):
        if value < 18:
            raise ValueError("Patient must be at least 18 years old")
        return value
    
    @model_validator(mode='after')
    def validate_emergency_contact(cls, model):
        if model.age > 60 and 'emergency_contact' not in model.contact_info:
            raise ValueError("Patients over 60 must have an emergency contact")
        return model
    
    @computed_field
    @property
    def bmi(self) -> float:
        return round(self.weight / ((self.height / 100) ** 2), 2)

BMI is also calculated

In [6]:
patient_info = {'name': 'benky', 'email': 'benky@sbi.com', 'linkedin_url': 'https://linkedin.com/in/benky', 'age': '65', 'weight': 70.5, 'height': 175.0, 'allergies': ['dust', 'pollen'], 'contact_info': {'phone': '1234567890', 'emergency_contact': '9876543210'}}
patient2 = Patient(**patient_info)
print(patient2)

name='BENKY' email='benky@sbi.com' linkedin_url=AnyUrl('https://linkedin.com/in/benky') age=65 weight=70.5 height=175.0 married=False allergies=['dust', 'pollen'] contact_info={'phone': '1234567890', 'emergency_contact': '9876543210'} bmi=23.02


Note the field name will be same as what you define the function name as.

Defined function name as bmi, thus got the field name also as bmi